# Week 4, Lab 2 — Chains and memory


In [ ]:
WEEK = 'Week 4'
LAB = 'Lab 2 — chains & memory'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate langchain langchain-huggingface langgraph
else:
    %pip install -q langchain langchain-ollama langgraph ollama


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

llm = get_langchain_llm()

outline = ChatPromptTemplate.from_template("List 3 bullet points outlining: {topic}")
draft = ChatPromptTemplate.from_template("Turn these bullets into 4 sentences:\n{bullets}")
outline_chain = outline | llm | StrOutputParser()
draft_chain = draft | llm | StrOutputParser()

bullets = outline_chain.invoke({"topic": "Model Context Protocol"})
print("OUTLINE:\n", bullets)
print("\nDRAFT:\n", draft_chain.invoke({"bullets": bullets}))

history = []
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a brief tutor."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chat = chat_prompt | llm | StrOutputParser()

def turn(user: str) -> str:
    answer = chat.invoke({"history": history, "input": user})
    history.append(HumanMessage(content=user))
    history.append(AIMessage(content=answer))
    return answer

print("\nT1:", turn("My name is Asha and I am learning agents."))
print("T2:", turn("What is my name and what am I learning?"))


If T2 forgets the name, the model is small or truncated — not broken memory code.
